# Unit Waveform Clustering

Analyze the spike waveforms of units in selected brain regions (e.g. **SI** = substantia innominata, **MA** = magnocellular nucleus) and use **unsupervised clustering** to test whether distinct populations (e.g. putative glutamatergic vs. GABAergic neurons) can be separated based on waveform shape.

**Workflow**
1. Load an ephys session and append CCF unit locations.
2. Select units in the target region(s).
3. For each unit, find the peak channel (largest trough-to-peak amplitude) and extract its mean waveform.
4. Extract shape features (trough-to-peak duration, half-width, peak/trough ratio, repolarization slope, ...).
5. Standardize features and estimate the number of clusters (elbow + silhouette).
6. Cluster with KMeans / Gaussian Mixture and visualize the resulting populations.

> Motivation: negative vs. positive correlation to value was observed for glutamatergic and GABAergic neurons in ventral pallidum — this notebook checks whether the two populations are separable purely from waveform morphology.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# Project utilities (src/aind_dft_ephys_analysis)
from nwb_utils import NWBUtils
from ephys_utils import append_units_locations, find_best_electrode
from ephys_behavior import get_units_passed_default_qc
from general_utils import find_ephys_sessions

%matplotlib inline

# ---- Configuration -------------------------------------------------------
# Waveform sampling rate (Neuropixels standard). Adjust if your probe differs.
SAMPLING_RATE_HZ = 30_000.0
MS_PER_SAMPLE = 1_000.0 / SAMPLING_RATE_HZ

# Brain regions to analyze
TARGET_REGIONS = ["SI", "MA"]

# Fixed window (in ms) extracted around each unit's trough so that waveforms
# from probes/sessions with different sample counts can be pooled together.
PRE_MS = 1.0
POST_MS = 2.0

RANDOM_STATE = 0


## 2. Load all sessions and pool SI/MA units

Loop over every available (spike-sorted) session and, for each unit that (a) passes default QC and (b) is located in `TARGET_REGIONS`, extract the peak-channel waveform.

For speed we load **only the ephys NWB** (`read_ephys_nwb`) — the behavior NWB is not needed for waveform analysis, and skipping `combine_nwb` avoids the slow behavior load/merge. To combine units across probes/sessions with different sample counts, each waveform is aligned to its trough and cut to a fixed window (`PRE_MS` before, `POST_MS` after the trough).


In [ ]:
# Discover all ephys sessions (prefer spike-sorted ones)
all_sessions, sessions_by_animal, spike_sorted_sessions = find_ephys_sessions()
sessions_to_use = spike_sorted_sessions if len(spike_sorted_sessions) else all_sessions
print(f"Found {len(all_sessions)} sessions; using {len(sessions_to_use)} spike-sorted sessions.")

# Fixed trough-aligned window (in samples). 0 ms == trough.
PRE_SAMPLES = int(round(PRE_MS / MS_PER_SAMPLE))
POST_SAMPLES = int(round(POST_MS / MS_PER_SAMPLE))
WIN_LEN = PRE_SAMPLES + POST_SAMPLES + 1
time_ms = (np.arange(WIN_LEN) - PRE_SAMPLES) * MS_PER_SAMPLE


def get_region_from_loc(loc):
    if loc is not None and isinstance(loc, dict):
        return loc.get('brain_region', None)
    return None


def extract_peak_window(wf):
    """Trough-aligned peak-channel trace of fixed length WIN_LEN, or None if invalid.

    wf : array (n_samples, n_channels) mean waveform for one unit.
    """
    if wf is None or not np.all(np.isfinite(wf)):
        return None
    troughs = wf.min(axis=0)
    peaks = wf.max(axis=0)
    best_ch = int(np.argmax(peaks - troughs))   # channel with largest trough-to-peak
    trace = wf[:, best_ch].astype(float)
    if np.ptp(trace) == 0:
        return None
    trough_idx = int(np.argmin(trace))
    # Pad with edge values so a fixed window centered on the trough always fits.
    padded = np.pad(trace, (PRE_SAMPLES, POST_SAMPLES), mode='edge')
    return padded[trough_idx:trough_idx + WIN_LEN]


# ---- Accumulate peak-channel waveforms across every session --------------
peak_waveforms = []   # (n_pooled, WIN_LEN)
kept_sessions = []    # session id per unit
kept_indices = []     # within-session unit index
kept_regions = []     # brain region per unit

for si, session_name in enumerate(sessions_to_use):
    try:
        # Load ONLY the ephys NWB (behavior not needed -> much faster than combine_nwb)
        nwb_data = NWBUtils.read_ephys_nwb(session_name=session_name)
        if nwb_data is None:
            print(f"[skip] {session_name}: no ephys NWB")
            continue
        session_id_clean = Path(nwb_data.session_id).stem
        nwb_data = append_units_locations(nwb_data, session_name=session_id_clean)
    except Exception as e:
        print(f"[skip] {session_name}: {e}")
        continue

    # Units passing the automated default QC (presence ratio, ISI violations,
    # amplitude cutoff) and not labeled 'noise'.
    try:
        qc_units = set(get_units_passed_default_qc(nwb_data).tolist())
    except Exception as e:
        print(f"[skip] {session_id_clean}: QC unavailable ({e})")
        continue

    waveform_mean = nwb_data.units['waveform_mean'][:]
    ccf_locations = nwb_data.units['ccf_location'][:]
    n_units = waveform_mean.shape[0]

    n_added = 0
    for u in range(n_units):
        if u not in qc_units:
            continue
        region = get_region_from_loc(ccf_locations[u])
        if region not in TARGET_REGIONS:
            continue
        trace = extract_peak_window(waveform_mean[u])
        if trace is None or len(trace) != WIN_LEN:
            continue
        peak_waveforms.append(trace)
        kept_sessions.append(session_id_clean)
        kept_indices.append(u)
        kept_regions.append(region)
        n_added += 1

    print(f"[{si + 1}/{len(sessions_to_use)}] {session_id_clean}: +{n_added} units "
          f"(pooled total {len(peak_waveforms)})")

    # Release file handle to keep memory/handles in check across many sessions.
    if getattr(nwb_data, 'io', None) is not None:
        try:
            nwb_data.io.close()
        except Exception:
            pass

peak_waveforms = np.vstack(peak_waveforms)
kept_regions = np.array(kept_regions)
kept_sessions = np.array(kept_sessions)
n_samples = peak_waveforms.shape[1]
print(f"\nPooled {peak_waveforms.shape[0]} QC-passing units across sessions in {TARGET_REGIONS}.")
print(f"Window: {WIN_LEN} samples ({PRE_MS} ms pre + {POST_MS} ms post trough).")


## 3. Overview of the pooled units


In [ ]:
print("Pooled units per region:")
for region, count in Counter(kept_regions).most_common():
    print(f"  {region}: {count}")

print("\nPooled units per session:")
for sess, count in Counter(kept_sessions).most_common():
    print(f"  {sess}: {count}")


## 4. Inspect the pooled peak-channel waveforms

Normalize each waveform (baseline-subtracted, trough-oriented negative, amplitude-normalized) for shape comparison across sessions.


In [ ]:
# Normalize each waveform for shape comparison: sign-align (trough negative) + amplitude-normalize.
def normalize_waveform(trace):
    trace = trace - np.median(trace[:max(1, int(0.1 * len(trace)))])  # baseline to pre-spike
    # Orient so the largest deflection (the spike) is negative (trough-first convention)
    if abs(trace.min()) < abs(trace.max()):
        trace = -trace
    peak_amp = np.abs(trace).max()
    if peak_amp > 0:
        trace = trace / peak_amp
    return trace

norm_waveforms = np.vstack([normalize_waveform(w) for w in peak_waveforms])

# Quick look at all normalized waveforms
plt.figure(figsize=(8, 5))
for w in norm_waveforms:
    plt.plot(time_ms, w, color='lightgrey', linewidth=0.5)
plt.plot(time_ms, norm_waveforms.mean(axis=0), color='C3', linewidth=2, label='mean')
plt.xlabel('Time (ms)')
plt.ylabel('Normalized amplitude')
plt.title(f'Peak-channel waveforms ({norm_waveforms.shape[0]} units)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Extract waveform shape features

Classic features used to separate narrow- vs. broad-spiking neurons:
- **trough_to_peak_ms**: time from the trough to the following positive peak
- **half_width_ms**: full width of the trough at half its minimum
- **peak_trough_ratio**: amplitude of the post-trough peak relative to the trough
- **repolarization_slope**: slope just after the trough
- **recovery_slope**: slope after the post-trough peak

In [ ]:
def extract_features(trace, time_ms):
    """Compute morphology features from a trough-aligned, normalized waveform."""
    n = len(trace)
    trough_idx = int(np.argmin(trace))
    trough_val = trace[trough_idx]

    # Post-trough peak (repolarization peak)
    if trough_idx < n - 1:
        rel_peak = int(np.argmax(trace[trough_idx:])) + trough_idx
    else:
        rel_peak = trough_idx
    peak_val = trace[rel_peak]

    trough_to_peak_ms = (rel_peak - trough_idx) * MS_PER_SAMPLE
    peak_trough_ratio = peak_val / (abs(trough_val) + 1e-12)

    # Half-width: width where trace <= half of trough depth
    half_level = trough_val / 2.0
    below = np.where(trace <= half_level)[0]
    half_width_ms = (below.max() - below.min()) * MS_PER_SAMPLE if below.size > 1 else 0.0

    # Repolarization slope: slope over ~0.15 ms right after trough
    win = max(2, int(0.15 / MS_PER_SAMPLE))
    end = min(n, trough_idx + win)
    if end - trough_idx >= 2:
        repolarization_slope = np.polyfit(time_ms[trough_idx:end], trace[trough_idx:end], 1)[0]
    else:
        repolarization_slope = 0.0

    # Recovery slope: slope over ~0.15 ms right after the post-trough peak
    end2 = min(n, rel_peak + win)
    if end2 - rel_peak >= 2:
        recovery_slope = np.polyfit(time_ms[rel_peak:end2], trace[rel_peak:end2], 1)[0]
    else:
        recovery_slope = 0.0

    return {
        'trough_to_peak_ms': trough_to_peak_ms,
        'half_width_ms': half_width_ms,
        'peak_trough_ratio': peak_trough_ratio,
        'repolarization_slope': repolarization_slope,
        'recovery_slope': recovery_slope,
    }


features = pd.DataFrame([extract_features(w, time_ms) for w in norm_waveforms])
features['session'] = kept_sessions
features['unit_index'] = kept_indices
features['region'] = kept_regions
features.head()


In [ ]:
feature_cols = [
    'trough_to_peak_ms',
    'half_width_ms',
    'peak_trough_ratio',
    'repolarization_slope',
    'recovery_slope',
]

# Distribution of the classic separator: trough-to-peak duration
plt.figure(figsize=(7, 4))
plt.hist(features['trough_to_peak_ms'], bins=30, color='C0', alpha=0.8)
plt.xlabel('Trough-to-peak duration (ms)')
plt.ylabel('Unit count')
plt.title('Trough-to-peak duration distribution')
plt.tight_layout()
plt.show()

## 6. Standardize features and estimate the number of clusters

In [ ]:
X = features[feature_cols].values
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_scaled = StandardScaler().fit_transform(X)

ks = range(2, 8)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), inertias, 'o-')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(list(ks), silhouettes, 'o-', color='C1')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette score')
plt.tight_layout()
plt.show()

best_k = list(ks)[int(np.argmax(silhouettes))]
print(f"Best k by silhouette: {best_k}")

## 7. Cluster the units

We fit both KMeans and a Gaussian Mixture Model. Set `N_CLUSTERS` from the diagnostics above (defaults to the silhouette-optimal `best_k`, expecting at least 2 populations).

In [ ]:
N_CLUSTERS = max(2, best_k)  # override manually if desired

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
gmm = GaussianMixture(n_components=N_CLUSTERS, random_state=RANDOM_STATE).fit(X_scaled)

features['cluster_kmeans'] = kmeans.labels_
features['cluster_gmm'] = gmm.predict(X_scaled)

# Use KMeans labels downstream (switch to 'cluster_gmm' if preferred)
labels = features['cluster_kmeans'].values
print(f"KMeans cluster sizes: {np.bincount(labels)}")
print(f"GMM cluster sizes:    {np.bincount(features['cluster_gmm'].values)}")

## 8. Visualize the clusters

In [ ]:
# 8a. Mean waveform per cluster
plt.figure(figsize=(8, 5))
for c in sorted(np.unique(labels)):
    mask = labels == c
    mean_wf = norm_waveforms[mask].mean(axis=0)
    plt.plot(time_ms, mean_wf, linewidth=2, label=f'Cluster {c} (n={mask.sum()})')
    for w in norm_waveforms[mask]:
        plt.plot(time_ms, w, color=f'C{c}', alpha=0.08, linewidth=0.5)
plt.xlabel('Time (ms)')
plt.ylabel('Normalized amplitude')
plt.title('Mean waveform per cluster')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8b. Feature space: trough-to-peak vs half-width, colored by cluster
plt.figure(figsize=(7, 6))
sc = plt.scatter(features['trough_to_peak_ms'], features['half_width_ms'],
                 c=labels, cmap='tab10', s=30, alpha=0.8)
plt.xlabel('Trough-to-peak duration (ms)')
plt.ylabel('Half-width (ms)')
plt.title('Waveform feature space by cluster')
plt.colorbar(sc, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8c. PCA projection of the standardized feature space
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=30, alpha=0.8)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.title('PCA of waveform features by cluster')
plt.colorbar(sc, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8d. Cluster composition by brain region
composition = pd.crosstab(features['region'], features['cluster_kmeans'])
print(composition)

composition.plot(kind='bar', stacked=True, figsize=(7, 4), colormap='tab10')
plt.xlabel('Brain region')
plt.ylabel('Unit count')
plt.title('Cluster composition per region')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 8e. Mean feature values per cluster (interpretation aid)
summary = features.groupby('cluster_kmeans')[feature_cols].mean()
summary['n_units'] = features.groupby('cluster_kmeans').size()
summary

## 9. Notes & next steps

- Clusters with **short trough-to-peak / narrow half-width** are typically fast-spiking (often putative GABAergic); **broad** waveforms are typically putative glutamatergic. Compare `summary` above against this expectation.
- To test the value-correlation hypothesis, cross-reference the `unit_index` in each cluster with your value-encoding analysis and check whether negative- vs. positive-correlated units segregate by cluster.
- Consider pooling waveforms across multiple sessions for a larger, more robust sample before drawing conclusions.
- `SAMPLING_RATE_HZ` is set to 30 kHz — confirm it matches your recording, since all duration features scale with it.